In [54]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV

In [55]:
import warnings
warnings.filterwarnings("ignore")

In [56]:
users=pd.read_csv('./Dataset/users.csv')
fusers=pd.read_csv('./Dataset/fusers.csv')

In [57]:
print("User shape: ",users.shape)
print("FUser shape: ",fusers.shape)

User shape:  (3474, 42)
FUser shape:  (3351, 38)


### Data Preprocessing

In [58]:
users.isnull().sum()

id                                       0
name                                     1
screen_name                              0
statuses_count                           0
followers_count                          0
friends_count                            0
favourites_count                         0
listed_count                             0
url                                   2208
lang                                     0
time_zone                              999
location                              1109
default_profile                       2442
default_profile_image                 3461
geo_enabled                           1319
profile_image_url                        0
profile_banner_url                     309
profile_use_background_image           390
profile_background_image_url_https       0
profile_text_color                       0
profile_image_url_https                  0
profile_sidebar_border_color             0
profile_background_tile               2167
profile_sid

In [59]:
isFake = np.ones(3351,int)
isNotFake = np.zeros(3474,int)

In [60]:
fusers["isFake"] = isFake
users["isFake"] = isNotFake

In [61]:
df_allUsers = pd.concat([fusers, users], ignore_index=True)
df_allUsers.columns = df_allUsers.columns.str.strip()

In [62]:
df_allUsers.head()

,id,name,screen_name,statuses_count,followers_count,friends_count,favourites_count,listed_count,created_at,url,...,notifications,description,contributors_enabled,following,updated,isFake,timestamp,crawled_at,test_set_1,test_set_2
0,80479674,YI YUAN,yi_twitts,29,19,255,1,0,Wed Oct 07 03:19:21 +0000 2009,http://www.jycondo.com,...,NaN,real estate sales,NaN,NaN,2013-06-12 18:38:35,1,NaN,NaN,NaN,NaN
1,82487179,Marcos Perez C,marcos_peca,1408,208,866,138,0,Wed Oct 14 23:40:17 +0000 2009,NaN,...,NaN,NaN,NaN,NaN,2013-06-12 18:38:35,1,NaN,NaN,NaN,NaN
2,105830531,curti lorenzo,curtilorenzo,39,59,962,8,0,Sun Jan 17 16:46:52 +0000 2010,http://www.valcavargna.com/,...,NaN,le corna del capro scappato dal gregge s'infil...,NaN,NaN,2013-06-12 18:38:35,1,NaN,NaN,NaN,NaN
3,114488344,ruben dario toscano,gatito2710,59,7,49,4,0,Mon Feb 15 15:49:58 +0000 2010,NaN,...,NaN,NaN,NaN,NaN,2013-06-12 18:38:35,1,NaN,NaN,NaN,NaN
4,123222267,Malek Khalaf,MalekKhalaf,987,60,521,61,1,Mon Mar 15 11:38:55 +0000 2010,http://www.facebook.com/Malek.AlBalawi,...,NaN,"MA student at JU, Interested in Juventus,Italy...",NaN,NaN,2013-06-11 17:39:44,1,NaN,NaN,NaN,NaN


In [63]:
df_allUsers = df_allUsers.sample(frac=1).reset_index(drop=True)

In [64]:
df_allUsers.shape

(6825, 43)

In [65]:
Y = df_allUsers.isFake
Y.shape

(6825,)

In [66]:
df_allUsers.columns

Index(['id', 'name', 'screen_name', 'statuses_count', 'followers_count',
       'friends_count', 'favourites_count', 'listed_count', 'created_at',
       'url', 'lang', 'time_zone', 'location', 'default_profile',
       'default_profile_image', 'geo_enabled', 'profile_image_url',
       'profile_banner_url', 'profile_use_background_image',
       'profile_background_image_url_https', 'profile_text_color',
       'profile_image_url_https', 'profile_sidebar_border_color',
       'profile_background_tile', 'profile_sidebar_fill_color',
       'profile_background_image_url', 'profile_background_color',
       'profile_link_color', 'utc_offset', 'is_translator',
       'follow_request_sent', 'protected', 'verified', 'notifications',
       'description', 'contributors_enabled', 'following', 'updated', 'isFake',
       'timestamp', 'crawled_at', 'test_set_1', 'test_set_2'],
      dtype='object')

In [67]:
df1=df_allUsers[['id','favourites_count','followers_count','statuses_count','friends_count','default_profile','default_profile_image','profile_use_background_image','utc_offset','listed_count','geo_enabled','lang','isFake']]

In [68]:
lang_list = list(enumerate(np.unique(df1["lang"])))
lang_dict = {name : i for i, name in lang_list}
print(lang_dict)

{'Select Language...': 0, 'ar': 1, 'da': 2, 'de': 3, 'el': 4, 'en': 5, 'en-AU': 6, 'en-GB': 7, 'en-gb': 8, 'es': 9, 'fil': 10, 'fr': 11, 'id': 12, 'it': 13, 'ja': 14, 'ko': 15, 'nl': 16, 'pl': 17, 'pt': 18, 'ru': 19, 'sv': 20, 'tr': 21, 'xx-lc': 22, 'zh-TW': 23, 'zh-cn': 24, 'zh-tw': 25}


In [69]:
z=df1.loc[:, "lang_num"] = df1["lang"].map(lambda x: lang_dict[x]).astype(int)

In [70]:
lang_list = list(enumerate(np.unique(df1["lang"])))
lang_dict = {name : i for i, name in lang_list}
df1.loc[:, "lang_num"] = df1["lang"].map(lambda x: lang_dict[x]).astype(int)

In [71]:
df1.drop(["lang"], axis=1, inplace=True)

In [72]:
df1.drop(["isFake"], axis=1, inplace=True)

In [73]:
df1.head()

,id,favourites_count,followers_count,statuses_count,friends_count,default_profile,default_profile_image,profile_use_background_image,utc_offset,listed_count,geo_enabled,lang_num
0,1664238487,14472,269,10126,183,NaN,NaN,NaN,-18000.0,30,NaN,5
1,359203826,0,9,0,458,1.0,NaN,1.0,NaN,0,NaN,9
2,1925081953,5596,370,3621,266,1.0,NaN,1.0,-18000.0,1,1.0,5
3,1604160445,13278,320,14210,193,NaN,NaN,NaN,-25200.0,1,1.0,5
4,19478719,2791,282,16484,306,NaN,NaN,1.0,28800.0,2,1.0,5


In [74]:
df1.isnull().sum()

id                                 0
favourites_count                   0
followers_count                    0
statuses_count                     0
friends_count                      0
default_profile                 2759
default_profile_image           6806
profile_use_background_image     399
utc_offset                      4015
listed_count                       0
geo_enabled                     4531
lang_num                           0
dtype: int64

In [75]:
df1 = df1.replace(np.nan, 0)

In [76]:
df1.isnull().sum()

id                              0
favourites_count                0
followers_count                 0
statuses_count                  0
friends_count                   0
default_profile                 0
default_profile_image           0
profile_use_background_image    0
utc_offset                      0
listed_count                    0
geo_enabled                     0
lang_num                        0
dtype: int64

### Post Preprocessing

In [77]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df1, Y, test_size=0.3,random_state=42)

In [78]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(4777, 12)
(2048, 12)
(4777,)
(2048,)


In [79]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import ShuffleSplit

In [80]:
cv=ShuffleSplit(n_splits=10,random_state=42,test_size=0.3)

### XGB Classifier

In [81]:
from xgboost import XGBClassifier
val_score=cross_val_score(XGBClassifier(),df1,Y,scoring='accuracy',cv=cv)

In [82]:
val_score

array([0.99365234, 0.99511719, 0.9921875 , 0.99169922, 0.99316406,
       0.99414062, 0.99316406, 0.9921875 , 0.99365234, 0.99169922])

In [83]:
np.mean(val_score)

np.float64(0.99306640625)

### Classifier

In [105]:
classifier = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    max_depth=7,
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    colsample_bytree=0.8,
    subsample=1.0
)
classifier.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [106]:
y_pred = classifier.predict(X_test)
predictions = [round(value) for value in y_pred]

In [107]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, predictions)
x=accuracy
print("Accuracy: %.2f%%" % (accuracy * 100.0))  

Accuracy: 99.32%


In [108]:
from sklearn.metrics import precision_score
precision_score(y_test, predictions)

0.9958289885297185

In [109]:
param_grid = {
    'n_estimators': [75,100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(),
    param_grid=param_grid,
    scoring='precision', 
    cv=5,
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 72 candidates, totalling 360 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.8, 1.0], 'learning_rate': [0.01, 0.1], 'max_depth': [3, 5, ...], 'n_estimators': [75, 100, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold an

In [110]:
best_model = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)

y_pred = best_model.predict(X_test)
print("Test Accuracy:", precision_score(y_test, y_pred))

Best Parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Test Accuracy: 0.9968684759916493


In [111]:
classifier.feature_names_in_

array(['id', 'favourites_count', 'followers_count', 'statuses_count',
       'friends_count', 'default_profile', 'default_profile_image',
       'profile_use_background_image', 'utc_offset', 'listed_count',
       'geo_enabled', 'lang_num'], dtype='<U28')

In [112]:
classifier.feature_importances_

array([0.03073309, 0.6274867 , 0.02956368, 0.18287301, 0.02080071,
       0.00326381, 0.01557487, 0.02036245, 0.00549024, 0.03788983,
       0.01638173, 0.0095799 ], dtype=float32)

In [113]:
import pickle

In [114]:
pickle.dump(classifier, open('profile_classifier.pkl', 'wb'))

In [122]:
classifier.feature_importances_

array([0.03073309, 0.6274867 , 0.02956368, 0.18287301, 0.02080071,
       0.00326381, 0.01557487, 0.02036245, 0.00549024, 0.03788983,
       0.01638173, 0.0095799 ], dtype=float32)

In [133]:
classifier.predict_proba(X_test.iloc[0].values.reshape(1,-1))

array([[3.4213066e-05, 9.9996579e-01]], dtype=float32)